# Rustren Studio — Animated Luma-Matte & Light-Leak Video Loops Baker (Colab GPU)

Pre-bakes **6 seamless 2.5s 720p H.264 video loops** (3 animated luma mattes + 3 animated optical overlays) + thumbnails + `video_catalog.json` using pure **PyTorch GPU shaders + FFmpeg** (zero `pip install`, ~15 seconds on T4/L4 GPU!).

In [ ]:
import os, json, math, zipfile, subprocess, torch
import numpy as np
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0) if device == "cuda" else "CPU"})')

W, H, FPS, FRAMES = 1280, 720, 30, 75  # 2.5 seconds
OUT_DIR = 'rustren_video_loops'
MEDIA_DIR = os.path.join(OUT_DIR, 'media')
THUMB_DIR = os.path.join(OUT_DIR, 'thumbs')
os.makedirs(MEDIA_DIR, exist_ok=True)
os.makedirs(THUMB_DIR, exist_ok=True)

# Normalized coordinate grids on GPU
yy, xx = torch.meshgrid(
    torch.linspace(-1.0, 1.0, H, device=device),
    torch.linspace(-1.0, 1.0, W, device=device),
    indexing='ij'
)

def fbm(x, y, t_phase):
    """Fast GPU fractal value-sine noise (seamless in t_phase 0..2pi)."""
    v = torch.zeros_like(x)
    amp = 0.5
    freq = 2.2
    for i in range(4):
        ang = t_phase * (1 if i % 2 == 0 else -1) + i * 1.7
        v = v + amp * torch.sin(x * freq + torch.cos(y * freq * 0.8 + ang)) * torch.cos(y * freq - torch.sin(x * freq * 0.7 - ang))
        freq *= 2.05
        amp *= 0.5
    return v

def render_video(vid_id, kind, title, tags, blend, track_matte, frame_fn):
    mp4_rel = f'media/{vid_id}.mp4'
    thumb_rel = f'thumbs/{vid_id}.jpg'
    mp4_path = os.path.join(OUT_DIR, mp4_rel)
    thumb_path = os.path.join(OUT_DIR, thumb_rel)

    cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{W}x{H}', '-r', str(FPS), '-i', '-',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '23', '-preset', 'medium', '-movflags', '+faststart',
        mp4_path
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    mid_frame = None
    for f in range(FRAMES):
        p = f / float(FRAMES)  # 0.0 .. <1.0
        rgb_t = frame_fn(p)    # (H, W, 3) float in 0..1 on GPU
        arr = (rgb_t.clamp(0.0, 1.0) * 255.0).to(torch.uint8).cpu().numpy()
        if f == FRAMES // 2:
            mid_frame = arr.copy()
        proc.stdin.write(arr.tobytes())
    proc.stdin.close()
    proc.wait()

    Image.fromarray(mid_frame, 'RGB').resize((320, 180), Image.Resampling.LANCZOS).save(thumb_path, 'JPEG', quality=82)
    kb = os.path.getsize(mp4_path) / 1024.0
    print(f'  [OK] {vid_id} ({kb:.0f} KB)')
    entry = {
        'id': vid_id,
        'kind': kind,
        'title': title,
        'tags': tags,
        'thumb': thumb_rel,
        'src': mp4_rel,
        'blend': blend,
        'is_video': True,
        'duration_sec': round(FRAMES / FPS, 2),
    }
    if track_matte:
        entry['track_matte'] = track_matte
    return entry

# 1. Organic Liquid Ink Wipe (Luma Matte)
def fn_ink_wipe(p):
    n = fbm(xx * 1.8, yy * 1.8, p * 2 * math.pi)
    edge = -1.35 + p * 2.7
    mask = torch.sigmoid((edge - (xx + n * 0.45)) * 18.0)
    return mask.unsqueeze(-1).repeat(1, 1, 3)

# 2. Soft Billowing Smoke Reveal (Luma Matte)
def fn_smoke_reveal(p):
    r = torch.sqrt(xx * xx * 0.7 + yy * yy)
    n = fbm(xx * 2.4, yy * 2.4, p * 2 * math.pi)
    radius = p * 1.55
    mask = torch.sigmoid((radius - (r + n * 0.35)) * 9.0)
    return mask.unsqueeze(-1).repeat(1, 1, 3)

# 3. Architectural Diagonal Blinds Wipe (Luma Matte)
def fn_diagonal_blinds(p):
    diag = (xx * 0.8 + yy * 0.5) * 4.0
    frac = diag - torch.floor(diag)
    mask = torch.sigmoid((p * 1.15 - frac) * 35.0)
    return mask.unsqueeze(-1).repeat(1, 1, 3)

# 4. Anamorphic Light Leak Sweep (Screen Overlay)
def fn_light_leak(p):
    ang = p * 2 * math.pi
    cx = math.sin(ang) * 0.85
    streak = torch.exp(-((yy * 5.5) ** 2)) * torch.exp(-(((xx - cx) * 1.4) ** 2))
    glow = torch.exp(-(((xx - cx) ** 2 + (yy * 1.2) ** 2) * 2.2)) * 0.7
    v = (streak + glow).clamp(0.0, 1.0)
    r = (v * 1.15).clamp(0, 1)
    g = (v * 0.72).clamp(0, 1)
    b = (v * 0.35).clamp(0, 1)
    return torch.stack([r, g, b], dim=-1)

# 5. Seamless Drifting Optical Bokeh (Screen Overlay)
rng = np.random.default_rng(77)
ORBS = [(rng.uniform(-1.0, 1.0), rng.uniform(-1.0, 1.0), rng.uniform(0.06, 0.19), rng.uniform(0.35, 0.85), rng.uniform(0, 2*math.pi)) for _ in range(18)]
def fn_bokeh_drift(p):
    ang = p * 2 * math.pi
    acc = torch.zeros_like(xx)
    for ox, oy, rad, bright, ph in ORBS:
        cx = ox + 0.14 * math.sin(ang + ph)
        cy = oy + 0.10 * math.cos(ang + ph * 1.3)
        d = torch.sqrt((xx - cx) ** 2 * 1.77 + (yy - cy) ** 2)
        orb = torch.sigmoid((rad - d) * 45.0) * (0.7 + 0.3 * torch.exp(-((d - rad * 0.85) * 25.0) ** 2))
        acc = acc + orb * bright * (0.75 + 0.25 * math.sin(ang + ph))
    return acc.clamp(0.0, 1.0).unsqueeze(-1).repeat(1, 1, 3)

# 6. Organic Cinema Film Burn (Screen Overlay)
def fn_film_burn(p):
    env = math.sin(p * math.pi) ** 1.4
    n = fbm(xx * 1.6 + p * 1.5, yy * 1.6, p * 2 * math.pi)
    flare = torch.exp(-((xx - (p * 1.8 - 0.9)) ** 2 * 1.8)) * (0.65 + 0.35 * n.clamp(0, 1)) * env
    r = (flare * 1.4).clamp(0, 1)
    g = (flare * 0.65).clamp(0, 1)
    b = (flare * 0.2).clamp(0, 1)
    return torch.stack([r, g, b], dim=-1)

print('Baking 6 GPU-shader video loops...')
catalog = [
    render_video('vid-ink-wipe', 'matte', 'Animated Ink Wipe', ['video', 'ink', 'matte'], 'normal', 'luma', fn_ink_wipe),
    render_video('vid-smoke-reveal', 'matte', 'Animated Smoke Reveal', ['video', 'smoke', 'matte'], 'normal', 'luma', fn_smoke_reveal),
    render_video('vid-diagonal-blinds', 'matte', 'Animated Diagonal Wipe', ['video', 'geometric', 'matte'], 'normal', 'luma', fn_diagonal_blinds),
    render_video('vid-light-leak', 'overlay', 'Animated Light Leak', ['video', 'flare', 'screen'], 'screen', None, fn_light_leak),
    render_video('vid-bokeh-drift', 'overlay', 'Animated Bokeh Drift', ['video', 'bokeh', 'screen'], 'screen', None, fn_bokeh_drift),
    render_video('vid-film-burn', 'overlay', 'Animated Film Burn', ['video', 'burn', 'screen'], 'screen', None, fn_film_burn),
]

with open(os.path.join(OUT_DIR, 'video_catalog.json'), 'w') as f:
    json.dump(catalog, f, indent=2)

zip_path = 'rustren_video_loops.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in sorted(files):
            full_p = os.path.join(root, fn)
            zf.write(full_p, os.path.relpath(full_p, OUT_DIR))

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'\nDONE! {len(catalog)} video loops baked into {zip_path} ({size_mb:.2f} MB)')

In [ ]:
from google.colab import files
files.download('rustren_video_loops.zip')